# Dynamic DeepHit: 5 final seeds, with vs. without actions in the input

Two variants, 5 seeds each:
- **with_actions**: fluid/vaso included in the inputs (latents intended for the RL *state*)
- **without_actions**: treatment removed from the inputs (health score intended for the RL *reward*)

This notebook generates train/val/test predictions for all 10 models (cached to each run's folder), evaluates discrimination/calibration across seeds, checks how consistent the reward is across seeds, and runs the vaso-onset event study.

In [1]:
import sys
import os
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

repo_root = Path("/home/jake/Documents/AI_Clinician/ai_clinician_pediatric_rl")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.supervised_learning.predict as ddh_predict
import src.supervised_learning.evaluate_dynamic_deephit_final_seeds as eval_ddh
import src.rl.evaluate_cql_alpha_sweep as eval_cql
import src.rl.evaluate_terminal_reward_models as etm

# Load models

In [2]:
unit = "picu"
illness = "sepsis"
hospital = "choc"
actions_included = ["with_actions", "without_actions"]

dynamic_deephit_model_dirs = {}
for actions in actions_included:
    dynamic_deephit_model_dir = f"/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/{hospital}/five_splits/split_0/sepsis/intake-vis/{unit}/{actions}"
    print(f"Using dynamic_deephit_model_dir for {actions}: {dynamic_deephit_model_dir}")
    dynamic_deephit_model_dirs[actions] = etm.get_model_dirs(dynamic_deephit_model_dir)



Using dynamic_deephit_model_dir for with_actions: /home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/with_actions
Found model directory: /home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/with_actions/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260925_181749_006821_job4167594.pbs-7/unknown_cx3-20-2_20260925-190322_26e7e2a7
Found model directory: /home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/with_actions/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard_split_0_20260925_181749_006821_job4167594.pbs-7/unknown_cx3-20-2_20260925-181827_9cc820b4
Found model directory: /home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/with_actions/choc_dynamic_deephit_transformer_discharge_mortality_tte_standard

In [3]:
# need to update paths in config_manager to point to local repo instead of rds
# swap the old rds cluster path prefix for the local path prefix in every config_manager attribute
old_prefix = "/rds/general/user/jormond/home/projects/ai_clinician"
new_prefix = "/home/jake/Documents/AI_Clinician"
def swap_prefix(value, label=""):
    if isinstance(value, str) and value.startswith(old_prefix):
        new_value = new_prefix + value[len(old_prefix):]
        # "code" and "data" folders don't exist in the local layout, so drop them
        new_value = "/".join(p for p in new_value.split("/") if p not in ("code", "data"))
        print(f"Updated {label}: {value} -> {new_value}")
        return new_value
    elif isinstance(value, dict):
        return {k: swap_prefix(v, f"{label}.{k}") for k, v in value.items()}
    elif isinstance(value, list):
        return [swap_prefix(v, f"{label}[{i}]") for i, v in enumerate(value)]
    else:
        return value

In [ ]:
ddh_models = {}
for actions in actions_included:
    # (seed, model_dir) pairs, sorted by seed so position = seed order
    seeded_dirs = sorted((eval_ddh.get_run_seed(d), d) for d in dynamic_deephit_model_dirs[actions])
    dynamic_deephit_model_dirs[actions] = [d for _, d in seeded_dirs]

    ddh_models[actions] = []
    for seed, model_dir in seeded_dirs:
        model, config_manager, hyperparams = eval_ddh.load_dynamic_deephit_model(
            model_dir, expected_actions_in_state=eval_ddh.VARIANTS[actions]
        )

        for attr_name, attr_value in vars(config_manager).items():
            setattr(config_manager, attr_name, swap_prefix(attr_value, attr_name))
        
        
        ddh_models[actions].append((model, config_manager, hyperparams, seed))
        print(f"Loaded {actions} seed {seed} from {model_dir}")

config_managers = {actions: models[0][1] for actions, models in ddh_models.items()}

Updated paths.experiments_dir: /rds/general/user/jormond/home/projects/ai_clinician/experiments -> /home/jake/Documents/AI_Clinician/experiments
Updated paths.supervised_final_model_dir: /rds/general/user/jormond/home/projects/ai_clinician/dynamic_deephit_results/choc/ -> /home/jake/Documents/AI_Clinician/dynamic_deephit_results/choc/
Updated paths.bc_transformer_final_model_dir: /rds/general/user/jormond/home/projects/ai_clinician/bc_transformer/choc/ -> /home/jake/Documents/AI_Clinician/bc_transformer/choc/
Updated paths.rl_models_dir: /rds/general/user/jormond/home/projects/ai_clinician/rl_models/choc/ -> /home/jake/Documents/AI_Clinician/rl_models/choc/
Updated paths.data_dir: /rds/general/user/jormond/home/projects/ai_clinician/data/choc_data -> /home/jake/Documents/AI_Clinician/choc_data
Updated paths.code_dir: /rds/general/user/jormond/home/projects/ai_clinician/code/choc_code -> /home/jake/Documents/AI_Clinician/choc_code
Updated _data_config.features_to_use_path: /rds/general/

# Load data

In [5]:
processors = {}
splits = ['train', 'val', 'test']
for split in splits:
    processors[split] = eval_ddh.load_dataset(config_manager, split=split)

# generate predictions

In [6]:
for actions in actions_included:
    eval_ddh.run_predictions(
        ddh_models[actions], processors,
        splits=("train", "val", "test"), batch_size=512, num_workers=4,
    )

train: 1680034 windows, 156 inputs


KeyboardInterrupt: 

# Config managers
Time bins are taken from `results.json` if saved there. Otherwise they come from the cohort/unit/filter_old lookup in the prediction branch of `supervised_learning.py`, which does **not** match the bins hardcoded in its `single_split` training branch; confirm which bins these models were trained with.

In [ ]:
all_run_dirs = [d for r in runs.values() for d in r.values()]
saved_time_bins = eval_ddh.training_time_bins(all_run_dirs)
print(f"Time bins from results.json: {saved_time_bins}")

TIME_BINS = saved_time_bins   # replace with the training bins if None, e.g. [0, 50, 103, 164, 237, 328, 460, 669, 1441]

config_managers = {
    variant: eval_ddh.build_config_manager(ARG_OVERRIDES, actions_in_state, time_bins=TIME_BINS)
    for variant, actions_in_state in eval_ddh.VARIANTS.items()
}
for variant, cm in config_managers.items():
    print(f"{variant}: actions_in_state={cm.training_config['actions_in_state']}, time_bins={cm.time_bins}")

# Sanity check: action features in the inputs
The action-free processor must contain no fluid/vaso-derived features (including `_mask` / `_delta` columns). The action names come from the with-actions `results.json`.

In [ ]:
first_with_run = next(iter(runs["with_actions"].values()))
ACTION_NAMES = eval_ddh.load_results_json(first_with_run)["hyperparameters"]["actions"]
print(f"Action names: {ACTION_NAMES}")

feature_names = {}
for variant, actions_in_state in eval_ddh.VARIANTS.items():
    print(f"\n{variant}:")
    feature_names[variant] = eval_ddh.check_action_features(
        config_managers[variant], ACTION_NAMES, expect_present=actions_in_state
    )

# Predictions
Generates train/val/test predictions for every seed (skipped where the `.h5` already exists). Train/val are written to disk for the RL pipeline; only test is kept in memory.

In [ ]:
test_predictions = {}
for variant in eval_ddh.VARIANTS:
    preds = eval_ddh.run_predictions(config_managers[variant], runs[variant], MODEL_TYPE)
    for seed in preds:
        test_predictions[(variant, seed)] = preds[seed]["test"]

In [ ]:
# test dataset (labels, trajectories) - shared by both variants, checked for alignment
test_dataset = eval_ddh.load_dataset(config_managers["with_actions"], split="test")
for key, preds in test_predictions.items():
    eval_ddh.check_alignment(preds, test_dataset)
print("All predictions aligned with the test dataset.")
index_ptr = np.asarray(test_dataset["index_ptr"])

# Discrimination and calibration (bootstrap)
Uses `bootstrap_evaluation_all_samples` rather than `predict.bootstrap_evaluation`, which saves only the last bootstrap sample. Resampling is by stay, since hourly rows within a stay are correlated. `seed_sd` is the between-seed spread; the pooled interval combines bootstrap and seed variability.

In [ ]:
N_BOOTSTRAP = 100
RESAMPLE_UNIT = "stay"

boot_by_model = {}
for (variant, seed), preds in test_predictions.items():
    print(f"Bootstrapping {variant} seed {seed}...")
    boot_by_model[(variant, seed)] = eval_ddh.bootstrap_evaluation_all_samples(
        preds, test_dataset,
        n_bootstrap_samples=N_BOOTSTRAP,
        resample_unit=RESAMPLE_UNIT,
        rng_seed=seed,
        save_dir=runs[variant][seed],
    )

boot_long, boot_summary = eval_ddh.summarise_bootstrap(boot_by_model)
boot_summary.to_csv(os.path.join(SAVE_DIR, f"bootstrap_summary_{RESAMPLE_UNIT}.csv"), index=False)

In [ ]:
# side-by-side comparison of the two variants
comparison = boot_summary.pivot(index="metric", columns="variant", values=["seed_mean", "seed_sd"])
comparison.columns = [f"{variant}_{stat}" for stat, variant in comparison.columns]
comparison["difference_without_minus_with"] = (
    comparison["without_actions_seed_mean"] - comparison["with_actions_seed_mean"]
)
pd.set_option("display.max_rows", 200)
comparison.round(4)

# Health score distributions per seed
Existing `plot_health_score_statistics` (death vs. survival KDEs and Mann-Whitney tests), saved to each run folder. Deaths should score lower; if not, check the head order assumed in `calculate_comprehensive_health_score`.

In [ ]:
for variant in eval_ddh.VARIANTS:
    for seed, run_dir in runs[variant].items():
        print(f"\n{variant} seed {seed}")
        ddh_predict.plot_health_score_statistics(eval_ddh.for_run(config_managers[variant], run_dir))

# Example patient trajectories
Writes ~100 plots per run, so restricted to selected seeds.

In [ ]:
PLOT_SEEDS = [list(runs["with_actions"])[0]]

for variant in eval_ddh.VARIANTS:
    for seed in PLOT_SEEDS:
        ddh_predict.plot_random_predictions(eval_ddh.for_run(config_managers[variant], runs[variant][seed]))

# Reward agreement across seeds
If the action-free rewards are highly correlated across seeds, the choice of reward seed matters little. The with vs. without comparison shows how much removing treatment changes the reward.

In [ ]:
health_scores = {key: np.asarray(p["health_score"]) for key, p in test_predictions.items()}
rewards = {key: eval_ddh.within_stay_reward(hs, index_ptr) for key, hs in health_scores.items()}

for name, series in [("health_score", health_scores), ("reward", rewards)]:
    pearson, spearman = eval_ddh.pairwise_agreement(series)
    pearson.to_csv(os.path.join(SAVE_DIR, f"{name}_pearson_across_models.csv"))
    spearman.to_csv(os.path.join(SAVE_DIR, f"{name}_spearman_across_models.csv"))
    fig = eval_ddh.plot_agreement(
        spearman, f"{name}: Spearman correlation across models (test)",
        save_path=os.path.join(SAVE_DIR, f"{name}_spearman_across_models.png"),
    )
    display(fig)

In [ ]:
# summary: mean off-diagonal correlation within each variant and between variants
_, reward_spearman = eval_ddh.pairwise_agreement(rewards)
labels = reward_spearman.index
is_with = labels.str.startswith("with_actions")
is_without = labels.str.startswith("without_actions")

def mean_offdiag(mask_rows, mask_cols):
    block = reward_spearman.values[np.ix_(mask_rows, mask_cols)]
    if np.array_equal(mask_rows, mask_cols):
        block = block[~np.eye(block.shape[0], dtype=bool)]
    return block.mean()

print(f"Reward Spearman, within with_actions:    {mean_offdiag(is_with, is_with):.3f}")
print(f"Reward Spearman, within without_actions: {mean_offdiag(is_without, is_without):.3f}")
print(f"Reward Spearman, between variants:       {mean_offdiag(is_with, is_without):.3f}")

# Vaso-onset event study
Hour 0 is the first hour with nonzero vaso. That dose is masked in the final input row, so the action-aware model first sees it at hour +1. A step at +1 present only in the with-actions curves is the input-leak signature; a decline that starts before hour 0 in both variants reflects the deterioration that prompted treatment (confounding by indication).

Onsets use the full test cohort of this dataset, not only sepsis stays. Check the printed firing fraction: zero dose is assumed to be the minimum of the normalised vaso column.

In [ ]:
print([f for f in feature_names["with_actions"] if any(f.startswith(a) for a in ACTION_NAMES)])
VASO_FEATURE = ACTION_NAMES[1]   # CHECK: the vaso dose column itself, not its _mask/_delta
PRE_HOURS, POST_HOURS = 12, 24

onsets = eval_ddh.find_vaso_onsets(test_dataset, VASO_FEATURE, min_pre_hours=PRE_HOURS)

In [ ]:
fig = eval_ddh.plot_event_study(
    health_scores, index_ptr, onsets,
    pre=PRE_HOURS, post=POST_HOURS,
    save_path=os.path.join(SAVE_DIR, "vaso_onset_event_study.png"),
)
fig